# Polars analysis caches

This notebook treats `trace.csv` and `summary.json` as stable CDT interchange files, then builds local Parquet caches for exploratory run debugging. The Parquet files live under `target/notebooks/analysis-caches/` and are disposable analysis artifacts, not part of the Rust crate API.

## 1. Setup

Run `just notebook-setup` before opening this notebook. The cells below find or build the `cdt` binary, run two moderate deterministic simulations, and keep all generated files under `target/`. This notebook is meant for debugging runs and is linted by CI, but it is heavier than the fast demo notebooks.

In [ ]:
import json
import math
import os
import subprocess
from dataclasses import asdict, dataclass
from pathlib import Path
from tempfile import mkdtemp
from typing import TYPE_CHECKING, Any, TypedDict, cast

import matplotlib.pyplot as plt
import polars as pl

if TYPE_CHECKING:
    from collections.abc import Callable

UTF8 = "utf-8"


class SlabTriangleProfileRow(TypedDict):
    run_id: str
    seed: int
    topology: str
    cosmological_constant: float
    temperature: float
    step: int
    slice: int
    triangles: int


class MeasurementRow(TypedDict):
    run_id: str
    seed: int
    topology: str
    cosmological_constant: float
    temperature: float
    step: int
    action: float
    vertices: int
    edges: int
    triangles: int
    slab_triangle_total: int


class RunSummaryRow(TypedDict):
    run_id: str
    seed: int
    topology: str
    cosmological_constant: float
    temperature: float
    steps: int
    thermalization_steps: int
    measurement_frequency: int
    acceptance_rate: float
    average_action: float
    elapsed_time_ms: float
    measurement_count: int
    step_count: int
    final_vertices: int
    final_edges: int
    final_triangles: int
    final_time_slices: int
    average_slab_triangle_total: float
    max_slab_triangle_fluctuation: float
    all_scale_effective_hausdorff_slope: float | None
    short_time_effective_spectral_dimension: float | None
    effective_dimensions_exported: bool


class CounterRow(TypedDict):
    run_id: str
    category: str
    counter: str
    value: int


@dataclass(frozen=True, slots=True)
class RunConfig:
    run_id: str
    seed: int
    topology: str = "open-boundary"
    cosmological_constant: float = 0.46209812037329684
    temperature: float = 1.0
    vertices_per_slice: int = 12
    timeslices: int = 10
    steps: int = 1_000
    thermalization_steps: int = 100
    measurement_frequency: int = 20


@dataclass(frozen=True, slots=True)
class RunArtifacts:
    trace_csv: Path
    summary: dict[str, Any]


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "Cargo.toml").is_file() and (path / "src" / "main.rs").is_file():
            return path
    message = "Run this notebook from inside the causal-triangulations repository."
    raise RuntimeError(message)


def run_command(command: list[str], *, cwd: Path, timeout: int = 300) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(  # noqa: S603 - this notebook intentionally invokes the repository binary with a fixed argv list.
        command,
        cwd=cwd,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.returncode != 0:
        command_text = " ".join(command)
        message = f"command failed with exit code {result.returncode}: {command_text}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        raise RuntimeError(message)
    return result


def cdt_command() -> list[str]:
    """Use the supplied executable or let Cargo select the build artifact."""
    configured = os.environ.get("CDT_BINARY")
    if configured is None:
        return ["cargo", "run", "--quiet", "--locked", "--release", "--bin", "cdt", "--"]
    binary = Path(configured).expanduser().resolve()
    if not binary.is_file():
        message = f"CDT_BINARY does not point to a file: {binary}"
        raise FileNotFoundError(message)
    return [str(binary)]


def run_directory(runs: Path, config: RunConfig) -> Path:
    runs = runs.resolve()
    destination = (runs / config.run_id).resolve()
    if destination == runs or not destination.is_relative_to(runs):
        raise ValueError(f"Run directory must be inside {runs}: {config.run_id!r}")
    return destination


ROOT = find_repo_root(Path.cwd().resolve())
CDT_COMMAND = cdt_command()

RUNS = (
    RunConfig(run_id="seed-105", seed=105),
    RunConfig(run_id="seed-106", seed=106, cosmological_constant=0.5),
)

print(f"Repository: {ROOT}")
print("CDT command:", *CDT_COMMAND)

## 2. Generate stable interchange files

The Rust binary writes CSV and JSON. Those files are the stable interchange boundary. Parquet appears only after the notebook reads and reshapes those outputs.

Each execution creates a fresh batch directory under `target/notebooks/analysis-caches/`. Its CSV, JSON and Parquet files stay together even when two kernels run the same configuration.

In [ ]:
def read_run_summary(config: RunConfig, path: Path) -> dict[str, Any]:
    loaded = json.loads(path.read_text(encoding=UTF8))
    if not isinstance(loaded, dict) or not isinstance(loaded.get("config"), dict):
        raise TypeError(f"{path} must contain a simulation configuration")
    expected = asdict(config)
    del expected["run_id"]
    expected["vertices"] = expected.pop("vertices_per_slice") * config.timeslices
    expected["dimension"] = 2
    for key, value in expected.items():
        actual = loaded["config"].get(key)
        numeric = key in {"temperature", "cosmological_constant"}
        valid_type = type(actual) in (int, float) if numeric else type(actual) is type(value)
        if not valid_type or actual != value:
            raise ValueError(f"{path} has {key}={actual!r}, expected {value!r} for {config.run_id}")
    return cast("dict[str, Any]", loaded)


def run_simulation(config: RunConfig, directory: Path) -> RunArtifacts:
    directory.mkdir(parents=True, exist_ok=True)
    trace_csv = directory / "trace.csv"
    summary_json = directory / "summary.json"
    command = [
        *CDT_COMMAND,
        "--dimension",
        "2",
        "--vertices-per-slice",
        str(config.vertices_per_slice),
        "--timeslices",
        str(config.timeslices),
        "--topology",
        config.topology,
        "--cosmological-constant",
        str(config.cosmological_constant),
        "--temperature",
        str(config.temperature),
        "--steps",
        str(config.steps),
        "--thermalization-steps",
        str(config.thermalization_steps),
        "--measurement-frequency",
        str(config.measurement_frequency),
        "--seed",
        str(config.seed),
        "--simulate",
        "--output-csv",
        str(trace_csv),
        "--output-json",
        str(summary_json),
    ]
    run_command(command, cwd=ROOT, timeout=900)
    return RunArtifacts(trace_csv, read_run_summary(config, summary_json))


ARTIFACTS: dict[RunConfig, RunArtifacts] = {}
batch_parent = ROOT / "target" / "notebooks" / "analysis-caches"
batch_parent.mkdir(parents=True, exist_ok=True)
batch_dir = Path(mkdtemp(prefix="batch-", dir=batch_parent))
CACHE_DIR = batch_dir / "cache"
directories = [run_directory(batch_dir / "runs", config) for config in RUNS]
if len({str(path).casefold() for path in directories}) != len(RUNS):
    message = "Each simulation needs a distinct output directory."
    raise ValueError(message)
CACHE_DIR.mkdir()
for config, directory in zip(RUNS, directories, strict=True):
    ARTIFACTS[config] = run_simulation(config, directory)
    print(f"{config.run_id}: {directory}")

## 3. Cache scalar traces as Parquet

Each trace row gets run metadata so multiple seeds, topologies, action parameters, chains, or proposal diagnostics can be compared without relying on directory names.

Runs can have different numbers of time slabs. The combined table keeps each exported slab column and uses null for slabs absent from a run.

In [ ]:
def read_trace_table(config: RunConfig, artifacts: RunArtifacts) -> pl.DataFrame:
    path = artifacts.trace_csv
    schema = {
        "step": pl.Int64,
        "accepted": pl.Boolean,
        "proposed": pl.Boolean,
        "action": pl.Float64,
        "vertices": pl.UInt32,
        "triangles": pl.UInt32,
        "edges": pl.UInt32,
        "log_prob": pl.Float64,
        "move_family": pl.UInt32,
        "seed_present": pl.UInt8,
        "seed_low_u32": pl.UInt32,
        "seed_high_u32": pl.UInt32,
    }
    trace = pl.read_csv(path, schema_overrides=schema)
    if missing := sorted(schema.keys() - set(trace.columns)):
        raise ValueError(f"{path} is missing required columns: {missing}")
    if any(trace.select(*schema).null_count().row(0)) or (trace["accepted"] & ~trace["proposed"]).any():
        raise ValueError(f"{path} contains missing observations or an accepted step without a proposal")
    if not all(trace.select(pl.col("action", "log_prob").is_finite().all()).row(0)) or 0 in trace.select("vertices", "edges", "triangles").min().row(0):
        message = "Trace requires finite scalar observations and positive simplex counts."
        raise ValueError(message)
    if not trace["step"].equals(pl.Series(range(1, config.steps + 1))):
        raise ValueError(f"{path} does not contain the requested simulation steps")
    seed_matches = (pl.col("seed_present") == 1) & (pl.col("seed_low_u32") == config.seed % 2**32) & (pl.col("seed_high_u32") == config.seed // 2**32)
    if not trace.select(seed_matches.all()).item():
        raise ValueError(f"{path} does not belong to seed {config.seed}")
    return trace.with_columns(
        pl.lit(config.run_id).alias("run_id"),
        pl.lit(config.seed).alias("seed"),
        pl.lit(config.topology).alias("topology"),
        pl.lit(config.cosmological_constant).alias("cosmological_constant"),
        pl.lit(config.temperature).alias("temperature"),
    )


trace_tables = [read_trace_table(config, ARTIFACTS[config]) for config in RUNS]
trace_cache = CACHE_DIR / "scalar_trace.parquet"
trace_table = pl.concat(trace_tables, how="diagonal_relaxed")
trace_table.write_parquet(trace_cache)

print(f"wrote local cache: {trace_cache}")
trace_table.select(
    pl.len().alias("rows"),
    pl.col("run_id").n_unique().alias("runs"),
    pl.col("accepted").mean().alias("overall_acceptance"),
    pl.col("action").mean().alias("mean_action"),
)

## 4. Cache run summaries and counters

`summary.json` carries run-level configuration, aggregate diagnostics, final counts, move statistics, proposal statistics, and scheduled measurements. The explicitly finite-window effective Hausdorff slope and spectral dimension are nullable here because the Rust API can compute them, but the current summary exporter does not serialize them yet.

In [ ]:
def parse_object(value: Any, *, name: str) -> dict[str, Any]:
    if isinstance(value, dict):
        return cast("dict[str, Any]", value)
    message = f"{name} should be an object: {value!r}"
    raise TypeError(message)


def parse_int(value: Any, *, name: str) -> int:
    if isinstance(value, int) and not isinstance(value, bool):
        return value
    message = f"{name} should be an integer: {value!r}"
    raise TypeError(message)


def parse_number(value: Any, *, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float):
        message = f"{name} should be numeric: {value!r}"
        raise TypeError(message)
    message = f"{name} should be finite: {value!r}"
    try:
        parsed = float(value)
    except OverflowError as error:
        raise ValueError(message) from error
    if not math.isfinite(parsed):
        raise ValueError(message)
    return parsed


def expect_rejection(action: Callable[[], object], expected: type[Exception], *, case: str) -> None:
    try:
        action()
    except expected:
        return
    message = f"numeric parser accepted invalid {case}"
    raise AssertionError(message)


expect_rejection(lambda: parse_int(value=True, name="self_check.integer"), TypeError, case="boolean integer")
expect_rejection(lambda: parse_number(value=False, name="self_check.number"), TypeError, case="boolean number")
expect_rejection(lambda: parse_number(value=10**400, name="self_check.number"), ValueError, case="overflowing integer")
for non_finite in (math.nan, math.inf, -math.inf):
    expect_rejection(
        lambda value=non_finite: parse_number(value, name="self_check.number"),
        ValueError,
        case=f"non-finite number {non_finite!r}",
    )


def parse_numeric_list(value: Any, *, name: str) -> list[float]:
    if not isinstance(value, list):
        message = f"{name} should be a list: {value!r}"
        raise TypeError(message)
    return [parse_number(item, name=f"{name}[{index}]") for index, item in enumerate(value)]


def optional_number(mapping: dict[str, Any], key: str) -> float | None:
    value = mapping.get(key)
    return None if value is None else parse_number(value, name=f"aggregate.{key}")


def run_summary_row(config: RunConfig, summary: dict[str, Any]) -> RunSummaryRow:
    aggregate = parse_object(summary.get("aggregate"), name="summary.aggregate")
    final = parse_object(summary.get("final_triangulation"), name="summary.final_triangulation")
    average_profile = parse_numeric_list(aggregate.get("average_slab_triangle_profile"), name="aggregate.average_slab_triangle_profile")
    fluctuations = parse_numeric_list(aggregate.get("slab_triangle_fluctuations"), name="aggregate.slab_triangle_fluctuations")
    hausdorff = optional_number(aggregate, "all_scale_effective_hausdorff_slope")
    spectral = optional_number(aggregate, "short_time_effective_spectral_dimension")
    return {
        "run_id": config.run_id,
        "seed": config.seed,
        "topology": config.topology,
        "cosmological_constant": config.cosmological_constant,
        "temperature": config.temperature,
        "steps": config.steps,
        "thermalization_steps": config.thermalization_steps,
        "measurement_frequency": config.measurement_frequency,
        "acceptance_rate": parse_number(aggregate.get("acceptance_rate"), name="aggregate.acceptance_rate"),
        "average_action": parse_number(aggregate.get("average_action"), name="aggregate.average_action"),
        "elapsed_time_ms": parse_number(aggregate.get("elapsed_time_ms"), name="aggregate.elapsed_time_ms"),
        "measurement_count": parse_int(aggregate.get("measurement_count"), name="aggregate.measurement_count"),
        "step_count": parse_int(aggregate.get("step_count"), name="aggregate.step_count"),
        "final_vertices": parse_int(final.get("vertices"), name="final.vertices"),
        "final_edges": parse_int(final.get("edges"), name="final.edges"),
        "final_triangles": parse_int(final.get("triangles"), name="final.triangles"),
        "final_time_slices": parse_int(final.get("time_slices"), name="final.time_slices"),
        "average_slab_triangle_total": math.fsum(average_profile),
        "max_slab_triangle_fluctuation": max(fluctuations, default=0.0),
        "all_scale_effective_hausdorff_slope": hausdorff,
        "short_time_effective_spectral_dimension": spectral,
        "effective_dimensions_exported": hausdorff is not None or spectral is not None,
    }


def counter_rows(config: RunConfig, summary: dict[str, Any]) -> list[CounterRow]:
    rows: list[CounterRow] = []
    for category in ("move_stats", "proposal_stats"):
        counters = parse_object(summary.get(category), name=f"summary.{category}")
        for counter in sorted(counters):
            rows.append(
                {
                    "run_id": config.run_id,
                    "category": category,
                    "counter": counter,
                    "value": parse_int(counters[counter], name=f"{category}.{counter}"),
                },
            )
    return rows


summary_rows = [run_summary_row(config, ARTIFACTS[config].summary) for config in RUNS]
counter_table = pl.DataFrame([row for config in RUNS for row in counter_rows(config, ARTIFACTS[config].summary)])
summary_table = pl.DataFrame(
    summary_rows,
    schema_overrides={"all_scale_effective_hausdorff_slope": pl.Float64, "short_time_effective_spectral_dimension": pl.Float64},
)

summary_cache = CACHE_DIR / "run_summary.parquet"
counter_cache = CACHE_DIR / "run_counters.parquet"
summary_table.write_parquet(summary_cache)
counter_table.write_parquet(counter_cache)
cached_counters = pl.read_parquet(counter_cache)

print(f"wrote local cache: {summary_cache}")
print(f"wrote local cache: {counter_cache}")
print(cached_counters.sort("run_id", "category", "counter"))
summary_table.select(
    "run_id",
    "temperature",
    "cosmological_constant",
    "acceptance_rate",
    "average_action",
    "final_vertices",
    "final_triangles",
    "all_scale_effective_hausdorff_slope",
    "short_time_effective_spectral_dimension",
    "effective_dimensions_exported",
)

## 5. Flatten slab-triangle profiles

Scheduled measurements and nested per-slab triangle profiles are cached separately. The long-form rows have shape `run_id, seed, topology, cosmological_constant, temperature, step, slice, triangles`, which is convenient for Polars group-by operations and matplotlib plots. These `N₂(t)` counts are distinct from configured spatial-vertex counts `N₀(t)`.

In [ ]:
def measurement_count(value: object, *, name: str, minimum: int = 0) -> int:
    """Decode a count using the bounds of the Rust measurement field."""
    count = parse_int(value, name=name)
    if not minimum <= count <= 0xFFFF_FFFF:
        message = f"{name} must be an integer between {minimum} and 4294967295"
        raise ValueError(message)
    return count


def measurement_rows(config: RunConfig, summary: dict[str, Any]) -> tuple[list[MeasurementRow], list[SlabTriangleProfileRow]]:
    """Build both tables from the same validated CDT measurement."""
    measurements = summary.get("measurements")
    if not isinstance(measurements, list):
        message = f"summary measurements should be a list for {config.run_id}"
        raise TypeError(message)

    rows: list[MeasurementRow] = []
    slabs: list[SlabTriangleProfileRow] = []
    for index, value in enumerate(measurements):
        field = f"measurement {index}"
        data = parse_object(value, name=field)
        step = measurement_count(data.get("step"), name=f"{field} step")
        triangles = measurement_count(data.get("triangles"), name=f"{field} triangles", minimum=1)
        profile = data.get("slab_triangle_profile")
        if not isinstance(profile, list):
            message = f"{field} slab_triangle_profile should be a list"
            raise TypeError(message)
        counts = [measurement_count(count, name=f"{field} slab {i}") for i, count in enumerate(profile)]
        total = sum(counts)
        if total > triangles:
            message = f"{field} slab counts sum to {total}, exceeding the triangle count {triangles}"
            raise ValueError(message)
        rows.append(
            {
                "run_id": config.run_id,
                "seed": config.seed,
                "topology": config.topology,
                "cosmological_constant": config.cosmological_constant,
                "temperature": config.temperature,
                "step": step,
                "action": parse_number(data.get("action"), name=f"{field} action"),
                "vertices": measurement_count(data.get("vertices"), name=f"{field} vertices", minimum=1),
                "edges": measurement_count(data.get("edges"), name=f"{field} edges", minimum=1),
                "triangles": triangles,
                "slab_triangle_total": total,
            },
        )
        for i, count in enumerate(counts):
            slabs.append(
                {
                    "run_id": config.run_id,
                    "seed": config.seed,
                    "topology": config.topology,
                    "cosmological_constant": config.cosmological_constant,
                    "temperature": config.temperature,
                    "step": step,
                    "slice": i,
                    "triangles": count,
                },
            )
    return rows, slabs


measurement_rows_all: list[MeasurementRow] = []
slab_triangle_rows: list[SlabTriangleProfileRow] = []
for config in RUNS:
    run_measurements, run_slabs = measurement_rows(config, ARTIFACTS[config].summary)
    measurement_rows_all.extend(run_measurements)
    slab_triangle_rows.extend(run_slabs)
measurement_schema = {
    "run_id": pl.String,
    "seed": pl.Int64,
    "topology": pl.String,
    "cosmological_constant": pl.Float64,
    "temperature": pl.Float64,
    "step": pl.Int64,
    "action": pl.Float64,
    "vertices": pl.Int64,
    "edges": pl.Int64,
    "triangles": pl.Int64,
    "slab_triangle_total": pl.Int64,
}
slab_triangle_schema = {
    "run_id": pl.String,
    "seed": pl.Int64,
    "topology": pl.String,
    "cosmological_constant": pl.Float64,
    "temperature": pl.Float64,
    "step": pl.Int64,
    "slice": pl.Int64,
    "triangles": pl.UInt32,
}
measurement_table = pl.DataFrame(measurement_rows_all, schema=measurement_schema)
slab_triangle_table = pl.DataFrame(slab_triangle_rows, schema=slab_triangle_schema)
measurement_cache = CACHE_DIR / "measurements.parquet"
slab_triangle_cache = CACHE_DIR / "slab_triangle_profile_long.parquet"
measurement_table.write_parquet(measurement_cache)
slab_triangle_table.write_parquet(slab_triangle_cache)
cached_measurements = pl.read_parquet(measurement_cache)

print(f"wrote local cache: {measurement_cache}")
print(f"wrote local cache: {slab_triangle_cache}")
cached_measurements.group_by("run_id").agg(
    pl.len().alias("measurements"),
    pl.col("action").mean().alias("mean_measured_action"),
    pl.col("slab_triangle_total").mean().alias("mean_slab_triangles"),
)

## 6. Plot cached analysis tables

The plots below read from Parquet caches. Delete the cache directory at any time; rerunning the notebook recreates it from the stable CSV/JSON outputs.

In [ ]:
cached_trace = pl.read_parquet(CACHE_DIR / "scalar_trace.parquet")
cached_slab_triangles = pl.read_parquet(CACHE_DIR / "slab_triangle_profile_long.parquet")
cached_summary = pl.read_parquet(CACHE_DIR / "run_summary.parquet")
trace_by_run = cached_trace.with_columns(pl.col("accepted").cast(pl.Int64).alias("accepted_int")).sort("run_id", "step")
slab_triangles_by_slice = cached_slab_triangles.group_by("run_id", "slice").agg(pl.col("triangles").mean().alias("mean_triangles")).sort("run_id", "slice")
run_ids = cached_summary.sort("run_id").get_column("run_id").to_list()

figure, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

for (run_id,), run_trace in trace_by_run.group_by("run_id", maintain_order=True):
    steps = run_trace["step"].to_list()
    running_acceptance = run_trace.select(pl.col("accepted_int").cum_sum() / pl.int_range(1, pl.len() + 1)).to_series().to_list()
    axes[0, 0].plot(steps, run_trace["action"].to_list(), label=run_id)
    axes[0, 1].plot(steps, run_trace["vertices"].to_list(), label=f"{run_id} vertices")
    axes[0, 1].plot(steps, run_trace["triangles"].to_list(), linestyle="--", label=f"{run_id} triangles")
    axes[0, 2].plot(steps, running_acceptance, label=run_id)

for (run_id,), run_profile in slab_triangles_by_slice.group_by("run_id", maintain_order=True):
    axes[1, 0].plot(run_profile["slice"].to_list(), run_profile["mean_triangles"].to_list(), marker="o", label=run_id)

axes[1, 1].bar(run_ids, cached_summary.sort("run_id").get_column("average_action").to_list())
axes[1, 2].bar(run_ids, cached_summary.sort("run_id").get_column("final_vertices").to_list(), label="vertices")
axes[1, 2].bar(run_ids, cached_summary.sort("run_id").get_column("final_triangles").to_list(), alpha=0.55, label="triangles")

axes[0, 0].set_title("Action trace")
axes[0, 0].set_xlabel("step")
axes[0, 0].set_ylabel("Regge action")

axes[0, 1].set_title("Volume counts")
axes[0, 1].set_xlabel("step")
axes[0, 1].set_ylabel("count")

axes[0, 2].set_title("Running acceptance")
axes[0, 2].set_xlabel("step")
axes[0, 2].set_ylabel("accepted / steps")

axes[1, 0].set_title("Mean slab-triangle profile")
axes[1, 0].set_xlabel("time slice")
axes[1, 0].set_ylabel("triangles")

axes[1, 1].set_title("Average measured action")
axes[1, 1].set_xlabel("run")
axes[1, 1].set_ylabel("action")

axes[1, 2].set_title("Final triangulation counts")
axes[1, 2].set_xlabel("run")
axes[1, 2].set_ylabel("count")

for axis in axes.ravel():
    if axis.get_legend_handles_labels()[0]:
        axis.legend()
    axis.grid(alpha=0.3)

plt.show()